# 第10回 解答

練習問題A, B（競争に出す自分の戦略）の解答はありません。以下では，本文の Bandit, MultiArmedBandit, RandomStrategy, TsuikiStrategy, MaxStrategy, OnePlay, MultiMABTester を使うので，先に本文と同じ定義を実行しておきます。

In [ ]:
import random
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt

class Bandit:
    def __init__(self):
        self.p = random.uniform(0,1)
    def slot(self):
        return 1 if random.uniform(0,1) < self.p else 0
    def update(self):
        if random.uniform(0,1) < 0.5:
            self.p = max(self.p - 0.05, 0)
        else:
            self.p = min(self.p + 0.05, 1)

class MultiArmedBandit:
    NUM = 10
    def __init__(self):
        self.bandits = [Bandit() for i in range(MultiArmedBandit.NUM)]
    def slot(self,i):
        return self.bandits[i].slot()
    def update(self):
        for b in self.bandits:
            b.update()
    def getmax(self):
        return max([x.p for x in self.bandits])
    def getmaxind(self):
        return max(range(len(self.bandits)), key=lambda i: self.bandits[i].p)

class RandomStrategy:
    def __init__(self, n):
        self.n = n
    def learn(self, k, result):
        return
    def choice(self):
        return random.randrange(self.n)
    def reset(self):
        return

class TsuikiStrategy:
    def __init__(self, n):
        self.n = n
        self.my = 0
    def learn(self, k, result):
        if result == 0:
            self.my = (self.my + 1) % self.n
    def choice(self):
        return self.my
    def reset(self):
        self.my = 0

class MaxStrategy:
    def __init__(self, m):
        self.mab = m
    def learn(self, k, result):
        return
    def choice(self):
        return self.mab.getmaxind()
    def reset(self):
        return

def OnePlay(numplay, strategies, mab, change=False):
    myaccounts = np.zeros(len(strategies), dtype=int)
    for x in strategies:
        x.reset()
    for i in range(numplay):
        for s in range(len(strategies)):
            k = strategies[s].choice()
            result = mab.slot(k)
            myaccounts[s] += result
            strategies[s].learn(k,result)
        if change:
            mab.update()
    return myaccounts/numplay

def MultiMABTester(n, strategyClasses, change=False, numplay=300):
    ret = np.zeros((len(strategyClasses)+1,0))
    for i in range(n):
        mab = MultiArmedBandit()
        strategies = [a(MultiArmedBandit.NUM) for a in strategyClasses]
        strategies.append(MaxStrategy(mab))
        ret = np.append(ret, OnePlay(numplay, strategies, mab, change).reshape(len(strategies),1), axis=1)
    return ret

In [ ]:
# 練習問題（いつも同じスロットマシンを引く戦略と比べる）
class ConstStrategy:
    ''' いつも 0 番を引く戦略 '''
    def __init__(self, n):
        self.n = n
    def learn(self, k, result):
        return
    def choice(self):
        return 0
    def reset(self):
        return

names = ["Const", "Random", "Tsuiki", "MAX"]
ret = MultiMABTester(1000, [ConstStrategy, RandomStrategy, TsuikiStrategy])
for name, r in zip(names, ret):
    print(f"{name:7}: 平均 {r.mean():.3f}, 標準偏差 {r.std():.3f}")

**考察**

- Const（いつも同じ台）と Random（ランダムに引く）は，どちらも平均の当たり確率は約 0.5 です（p の平均が 0.5 なので）。ただし，Const は1台の p だけで結果が決まるので，ゲームごとのばらつきがとても大きく（p が一様分布なら標準偏差は約 0.29），Random は 10 台の平均になるので，ばらつきが小さくなります。
- Tsuiki（当たったら同じ台，外れたら次の台）は，学習をしている分，平均が 0.7 を超えます。確率の高い台では当たりが続いて長く留まり，確率の低い台はすぐに離れるからです。ただし，確率が最高の台にずっと留まるわけではないので，MAX には届きません。
- どの戦略がよいかは，目的によります。平均を上げたいなら Tsuiki，結果のばらつきを小さくしたい（大負けしたくない）なら Random の方が Const より安全です。

In [ ]:
# 練習問題（最大の期待値に対する当たり回数の比の平均と分散）
# 1ゲームを 1000 回引くことにして，1000 ゲーム行う（時間がかかる場合は数を減らそう）
def ratios(strategyClass, games=1000, numplay=1000):
    r = []
    for g in range(games):
        mab = MultiArmedBandit()
        s = strategyClass(MultiArmedBandit.NUM)
        score = OnePlay(numplay, [s], mab)[0]
        r.append(score / mab.getmax())     # 最高の台を引き続けた時の期待値に対する比
    return np.array(r)

fig, ax = plt.subplots()
for sc in [RandomStrategy, TsuikiStrategy]:
    r = ratios(sc)
    print(f"{sc.__name__}: 平均 {r.mean():.3f}, 分散 {r.var():.4f}")
    ax.hist(r, bins=40, alpha=0.5, label=sc.__name__)
ax.legend()
plt.show()

In [ ]:
# 練習問題（change=True でばらつきが小さくなる理由）
# 300 回の間に，10 台のスロットマシンの確率がどう変化するかをグラフにする
mab = MultiArmedBandit()
history = []
for t in range(300):
    history.append([b.p for b in mab.bandits])
    mab.update()
history = np.array(history)

fig, ax = plt.subplots()
ax.plot(history)
ax.plot(history.max(axis=1), color="black", linewidth=2, label="max")
ax.set_xlabel("t")
ax.set_ylabel("p")
ax.legend()
plt.show()

**考察**

それぞれの台の確率は，1 回ごとに ±0.05 ずつ動くので，300 回の間に（酔歩として）$0.05 \times \sqrt{300} \approx 0.87$ 程度も動きます。そのため，最初にどの台の確率が高かったかは，ゲームの途中ですぐに意味を失い，台どうしの順位が何度も入れ替わります。

change=False の時は，最初に配られた 10 台の確率（特にその最大値）がゲーム全体の成績を決めるので，配られ方の運によって成績が大きくばらつきます。change=True の時は，300 回の間にいろいろな確率の状態を経験するので，その運が平均化され，ゲームごとの成績のばらつきが小さくなります。また，確率が 0 や 1 の近くにたまりやすいので，一番よい台の確率は高くなり，MAX の成績は少し上がります。

In [ ]:
# 確認：change=False と change=True での，ゲームごとの成績の標準偏差
for ch in [False, True]:
    ret = MultiMABTester(300, [TsuikiStrategy, RandomStrategy], change=ch)
    print(f"change={ch}: " + ", ".join(f"{name} 平均 {r.mean():.3f} 標準偏差 {r.std():.3f}" for name, r in zip(["Tsuiki", "Random", "MAX"], ret)))